# 09 — Hybrid Forecast Models

This notebook combines saved ARIMA, Random Forest, and XGBoost predictions. It does not retrain
any component model. Non-negative weights summing to one are selected on 2021 validation data
only and applied unchanged to 2022.


## 1. Load protocol, saved predictions, and verify keys


In [ ]:
from pathlib import Path
import importlib.util
import json
import pickle
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook", rc={"figure.dpi": 120, "savefig.dpi": 300})

def locate_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "phase4_utils.py").exists():
            return candidate
    raise FileNotFoundError("Project root not found.")

PROJECT_ROOT = locate_root(Path.cwd())
spec = importlib.util.spec_from_file_location("phase4_utils", PROJECT_ROOT / "src" / "phase4_utils.py")
u = importlib.util.module_from_spec(spec)
spec.loader.exec_module(u)
evaluation = u.load_evaluation_module(PROJECT_ROOT)
splits = u.load_splits(PROJECT_ROOT)
print({name: data["combined"].shape for name, data in splits.items()})


In [ ]:
for split_name, data in splits.items():
    assert data["X"][u.KEYS].equals(data["y"][u.KEYS])
    assert sorted(data["X"]["year"].unique().tolist()) == u.SPLIT_YEARS[split_name]
    assert data["X"]["region"].nunique() == 13
    assert not data["X"].duplicated(u.KEYS).any()
    assert list(data["X"].columns) == u.KEYS + u.PREDICTORS
print("Frozen protocol and target alignment verified.")


In [ ]:
MODEL_DIR=PROJECT_ROOT/"models"/"hybrid"; RESULT_DIR=PROJECT_ROOT/"results"/"hybrid"
FIGURE_DIR=PROJECT_ROOT/"figures"/"hybrid"
for directory in (MODEL_DIR,RESULT_DIR,FIGURE_DIR): directory.mkdir(parents=True,exist_ok=True)
u.write_json(RESULT_DIR/"environment.json",u.package_versions(
 ["numpy","pandas","matplotlib","seaborn"]))

paths={
 "ARIMA":PROJECT_ROOT/"results"/"arima"/"predictions.csv",
 "Random Forest":PROJECT_ROOT/"results"/"random_forest"/"predictions.csv",
 "XGBoost":PROJECT_ROOT/"results"/"xgboost"/"predictions.csv",
}
missing=[str(path.relative_to(PROJECT_ROOT)) for path in paths.values() if not path.exists()]
if missing: raise FileNotFoundError(f"Run Notebooks 06–08 first. Missing: {missing}")

components={}
required=["region","year","actual","predicted","model_name","dataset_split"]
for name,path in paths.items():
 frame=pd.read_csv(path)
 if any(column not in frame for column in required): raise ValueError(f"{name} prediction schema invalid")
 if frame.duplicated(["region","year","dataset_split"]).any(): raise ValueError(f"{name} duplicate keys")
 components[name]=frame.sort_values(["dataset_split","year","region"]).reset_index(drop=True)

reference=components["ARIMA"][["region","year","actual","dataset_split"]]
for name,frame in components.items():
 if not reference.equals(frame[["region","year","actual","dataset_split"]]):
  raise ValueError(f"{name} keys, split labels, or actual values differ.")
print("All component prediction keys and actual values are identical.")


## 2. Validation-only non-negative weight selection


In [ ]:
wide=reference.copy()
for name,frame in components.items(): wide[name]=frame["predicted"].to_numpy()

COMBINATIONS=[
 ("ARIMA + Random Forest",["ARIMA","Random Forest"]),
 ("ARIMA + XGBoost",["ARIMA","XGBoost"]),
 ("ARIMA + Random Forest + XGBoost",["ARIMA","Random Forest","XGBoost"]),
]

def candidate_weights(n):
 if n==2:
  return [np.array([w,1-w]) for w in np.linspace(0,1,101)]
 values=np.linspace(0,1,21)
 return [np.array([a,b,1-a-b]) for a in values for b in values if a+b<=1+1e-12]

validation=wide.query("dataset_split == 'validation'")
weight_rows=[]
for hybrid_name,names in COMBINATIONS:
 best=None
 matrix=validation[names].to_numpy(float)
 for weights in candidate_weights(len(names)):
  predicted=matrix@weights
  metric=evaluation.evaluate_regression(validation["actual"],predicted)
  candidate=(metric["RMSE"],metric["MAE"],weights)
  if best is None or candidate[0:2]<best[0:2]: best=candidate
 record={"hybrid_model":hybrid_name,"validation_RMSE":best[0],"validation_MAE":best[1]}
 record.update({f"weight_{name.lower().replace(' ','_')}":float(best[2][i])
                for i,name in enumerate(names)})
 weight_rows.append(record)
weights_table=pd.DataFrame(weight_rows)
display(weights_table)


## 3. Apply selected weights once to validation and test


In [ ]:
outputs=[]; metric_rows=[]
for hybrid_name,names in COMBINATIONS:
 row=weights_table.query("hybrid_model == @hybrid_name").iloc[0]
 weights=np.array([row[f"weight_{name.lower().replace(' ','_')}"] for name in names])
 if (weights<0).any() or not np.isclose(weights.sum(),1): raise ValueError("Invalid hybrid weights")
 for split_name in ["validation","test"]:
  subset=wide.query("dataset_split == @split_name").copy()
  predicted=subset[names].to_numpy(float)@weights
  output=u.prediction_frame(subset[u.KEYS],subset["actual"],predicted,hybrid_name,split_name)
  outputs.append(output); metric_rows.append(u.evaluate_prediction_frame(evaluation,output))
hybrid_predictions=pd.concat(outputs,ignore_index=True)
hybrid_metrics=pd.DataFrame(metric_rows).sort_values(["split","RMSE","model"]).reset_index(drop=True)
hybrid_predictions.to_csv(RESULT_DIR/"predictions.csv",index=False,float_format="%.15g")
hybrid_metrics.to_csv(RESULT_DIR/"metrics.csv",index=False,float_format="%.15g")
weights_table.to_csv(RESULT_DIR/"selected_weights.csv",index=False,float_format="%.15g")
u.write_json(MODEL_DIR/"selected_weights.json",json.loads(weights_table.fillna(0).to_json(orient="records")))
display(hybrid_metrics)


## 4. Hybrid figures


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5.5))
for ax,split_name in zip(axes,["validation","test"]):
 group=hybrid_predictions.query("dataset_split == @split_name")
 actual=group[["region","actual"]].drop_duplicates().sort_values("actual")
 ax.plot(actual["actual"].to_numpy()/1e9,marker="o",label="Observed",linewidth=2.5)
 for model_name,subset in group.groupby("model_name"):
  values=subset.set_index("region").loc[actual["region"]]
  ax.plot(values["predicted"].to_numpy()/1e9,marker=".",label=model_name)
 ax.set_title(split_name.title()); ax.set_ylabel("Billion kWh")
axes[1].legend(bbox_to_anchor=(1.02,1),loc="upper left",frameon=False)
fig.tight_layout(); fig.savefig(FIGURE_DIR/"predictions.png",bbox_inches="tight"); plt.show()

fig,ax=plt.subplots(figsize=(10,5.5)); test_metrics=hybrid_metrics.query("split == 'test'").copy()
sns.barplot(data=test_metrics,x=test_metrics["RMSE"]/1e9,y="model",color="#C8553D",ax=ax)
ax.set_xlabel("Test RMSE (billion kWh)"); ax.set_ylabel(""); ax.set_title("Hybrid Test Errors",weight="bold")
fig.tight_layout(); fig.savefig(FIGURE_DIR/"model_comparison.png",bbox_inches="tight"); plt.show()

plot_weights=weights_table.melt(id_vars=["hybrid_model"],value_vars=[
 c for c in weights_table if c.startswith("weight_")],var_name="component",value_name="weight").fillna(0)
fig,ax=plt.subplots(figsize=(11,6)); sns.barplot(data=plot_weights,x="weight",y="hybrid_model",
 hue="component",ax=ax); ax.set_xlim(0,1); ax.set_title("Validation-Selected Hybrid Weights",weight="bold")
fig.tight_layout(); fig.savefig(FIGURE_DIR/"selected_weights.png",bbox_inches="tight"); plt.show()


## 5. Reports and Phase 4 manifest


In [ ]:
report='''# Hybrid Model Summary

Weights are non-negative, sum to one, and were selected using 2021 validation observations
only. The selected weights were applied unchanged to 2022. Component models were not retrained
in this notebook. With only 13 validation observations, weights are uncertain and may be
unstable; test results are preserved regardless of performance.
'''
(RESULT_DIR/"summary_report.md").write_text(report,encoding="utf-8")

summary_path=PROJECT_ROOT/"results"/"phase4_validation_summary.md"
summary=f'''# Phase 4 Validation Summary

- Notebooks created: 06_ARIMA_Model, 07_Random_Forest_Model, 08_XGBoost_Model, 09_Hybrid_Model
- Frozen input protocol: 2019–2020 train, 2021 validation, 2022 test
- Forecast horizon: one year
- Evaluation metrics: MAE, RMSE, MAPE, R² via `src/evaluation.py`
- Hybrid weights: validation-only, non-negative, sum to one
- Phase boundary: no final comparison, final selection, or final forecast notebook created
- Warnings: very small sample, short histories, substantial regional variability; ARIMA fit
  failures (if any) are recorded in diagnostics
- Execution status: all four notebooks completed
- Frozen Phase 1–3 artifacts: not modified by these notebooks
'''
summary_path.write_text(summary,encoding="utf-8")

generated_roots=[
 PROJECT_ROOT/"notebooks",PROJECT_ROOT/"models"/"arima",PROJECT_ROOT/"models"/"random_forest",
 PROJECT_ROOT/"models"/"xgboost",PROJECT_ROOT/"models"/"hybrid",
 PROJECT_ROOT/"results"/"arima",PROJECT_ROOT/"results"/"random_forest",
 PROJECT_ROOT/"results"/"xgboost",PROJECT_ROOT/"results"/"hybrid",
 PROJECT_ROOT/"figures"/"arima",PROJECT_ROOT/"figures"/"random_forest",
 PROJECT_ROOT/"figures"/"xgboost",PROJECT_ROOT/"figures"/"hybrid"]
notebook_names={f"{i:02d}_":f"{i:02d}" for i in range(6,10)}
rows=[]
for base in generated_roots:
 if not base.exists(): continue
 for path in sorted(base.rglob("*")):
  if not path.is_file() or path.name=="phase4_output_manifest.csv": continue
  if base.name=="notebooks" and not path.name.startswith(tuple(notebook_names)): continue
  relative=path.relative_to(PROJECT_ROOT)
  generator=("09_Hybrid_Model.ipynb" if "hybrid" in path.parts or path==summary_path
   else "06_ARIMA_Model.ipynb" if "arima" in path.parts
   else "07_Random_Forest_Model.ipynb" if "random_forest" in path.parts
   else "08_XGBoost_Model.ipynb" if "xgboost" in path.parts
   else path.name)
  row_count=column_count=""
  if path.suffix.lower()==".csv":
   table=pd.read_csv(path); row_count=len(table); column_count=table.shape[1]
  rows.append({"output_file":str(relative),"generating_notebook":generator,
   "file_type":path.suffix.lower().lstrip(".") or "file","row_count":row_count,
   "column_count":column_count,"sha256":u.sha256(path)})
if summary_path.exists():
 rows.append({"output_file":str(summary_path.relative_to(PROJECT_ROOT)),
  "generating_notebook":"09_Hybrid_Model.ipynb","file_type":"md","row_count":"",
  "column_count":"","sha256":u.sha256(summary_path)})
for path, generator in [
  (PROJECT_ROOT / "src" / "phase4_utils.py", "Phase 4 shared utility"),
  (PROJECT_ROOT / "environment.yml", "Project environment specification"),]:
 if path.exists():
  rows.append({"output_file":str(path.relative_to(PROJECT_ROOT)),
   "generating_notebook":generator,"file_type":path.suffix.lower().lstrip(".") or "file",
   "row_count":"","column_count":"","sha256":u.sha256(path)})
manifest=pd.DataFrame(rows).drop_duplicates("output_file").sort_values("output_file")
manifest.to_csv(PROJECT_ROOT/"results"/"phase4_output_manifest.csv",index=False)
print("Notebook 09 and Phase 4 manifest complete.")
